# Getting [MASK] probabilities from BERT (for Diffusion Models)

**Ⓒ 2026 by [Damir Cavar](http://damir.cavar.me/)**

**Download:** This and various other Jupyter notebooks are available from my [GitHub repo](https://github.com/dcavar/python-tutorial-notebooks).

**Version:** 0.1, March 2026

**License:** [Creative Commons Attribution-ShareAlike 4.0 International License](https://creativecommons.org/licenses/by-sa/4.0/) ([CA BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/))

This example code is related to the discussion of DiffusionBERT and diffusion-based language models in my [Advanced Machine Learning](https://damir.cavar.me/l675/) class, Spring 2026 at Indiana University Bloomington.

For diffusion and other related computations it might be helpful to pull predictions of tokens in [MASK]-ed positions in text. This is an example that shows how this can be done.

In [15]:
import torch
from transformers import BertTokenizer, BertForMaskedLM
import torch.nn.functional as F

Loading pre-trained uncased BERT and tokenizer:

In [16]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)
# model.eval()

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Prepare a masked token in text and get the tensors for each token:

In [17]:
text = "Richard Feynman pineered work in [MASK] electrodynamics."
inputs = tokenizer(text, return_tensors="pt")

Get model predictions / logits:

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

Find the position of the [MASK] token:

In [20]:
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

Extract logits for the mask position and apply Softmax to get probabilities:

In [21]:
mask_logits = logits[0, mask_token_index, :]
probabilities = F.softmax(mask_logits, dim=-1) # use the last element of tensor (across vocabulary)

Get the top N most likely words:

In [24]:
n = 3

In [25]:
top_n_results = torch.topk(probabilities, n, dim=-1)
top_n_probs = top_n_results.values[0]
top_n_indices = top_n_results.indices[0]

In [26]:
for i in range(n):
    token = tokenizer.convert_ids_to_tokens([top_n_indices[i]])[0]
    prob = top_n_probs[i].item()
    print(f"Token: {token:<10} | Probability: {prob:.4f}")

Token: quantum    | Probability: 0.6016
Token: theoretical | Probability: 0.0358
Token: applied    | Probability: 0.0233


Ⓒ 2026 by [Damir Cavar](http://damir.cavar.me/) - [Creative Commons Attribution-ShareAlike 4.0 International License](https://creativecommons.org/licenses/by-sa/4.0/) ([CA BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/))